# Week 4 — Streamlit Deployment & Final Polish

This notebook builds, documents, and validates the interactive dashboard **before** deployment.

**Important:** the actual dashboard (`app.py`) is a standalone Streamlit script — it does **not** run inside notebook cells. This notebook's job is to:
1. Confirm every input file the app depends on is present and correctly structured
2. Write `app.py` and `requirements.txt` to disk
3. Test the app's core logic (data loading, filtering, confidence-interval math) here, where errors are easy to see and fix
4. Document how to run and deploy the app

**Files this app depends on** (all from Week 1-3, must sit in the same folder as `app.py`):
- `monthly_demand_clean.csv` — historical demand (Week 1)
- `final_anomaly_dataset.csv` — anomaly flags (Week 2)
- `future_demand_forecast.csv` — forecast + 95% CI (Week 3)
- `model_comparison_results.csv`, `best_model_per_category.csv` — model evaluation (Week 3)

In [1]:
import os
import pandas as pd
import numpy as np

REQUIRED_FILES = [
    'monthly_demand_clean.csv',
    'final_anomaly_dataset.csv',
    'future_demand_forecast.csv',
    'model_comparison_results.csv',
    'best_model_per_category.csv',
]

print("Working directory:", os.getcwd())
missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
if missing:
    print("MISSING FILES (copy these into this folder before continuing):")
    for f in missing:
        print("  -", f)
else:
    print("All required files found.")

Working directory: C:\Users\ASUS\Downloads\Week4-Streamlit-App
All required files found.


## Step 1 — Inspect each file's structure
Confirms column names match exactly what `app.py` expects. This is the most common source of bugs when wiring a dashboard to files produced across different weeks/teammates.

In [2]:
historical_df = pd.read_csv('monthly_demand_clean.csv', index_col=0, parse_dates=True)
print('monthly_demand_clean.csv ->', historical_df.shape, list(historical_df.columns))

anomalies_df = pd.read_csv('final_anomaly_dataset.csv', parse_dates=['OrderDate'])
print('final_anomaly_dataset.csv ->', anomalies_df.shape, list(anomalies_df.columns))

future_df = pd.read_csv('future_demand_forecast.csv', parse_dates=['OrderDate'])
print('future_demand_forecast.csv ->', future_df.shape, list(future_df.columns))

model_comparison_df = pd.read_csv('model_comparison_results.csv')
print('model_comparison_results.csv ->', model_comparison_df.shape, list(model_comparison_df.columns))

best_model_df = pd.read_csv('best_model_per_category.csv')
print('best_model_per_category.csv ->', best_model_df.shape, list(best_model_df.columns))

monthly_demand_clean.csv -> (54, 5) ['CPU', 'Mother Board', 'RAM', 'Storage', 'Video Card']
final_anomaly_dataset.csv -> (270, 11) ['OrderDate', 'CategoryName', 'Quantity', 'ZScore', 'ZScore_Anomaly', 'IQR_Lower', 'IQR_Upper', 'IQR_Anomaly', 'IF_Anomaly', 'Anomaly_Method_Count', 'Final_Anomaly']
future_demand_forecast.csv -> (15, 5) ['OrderDate', 'CategoryName', 'Forecast', 'Lower_CI', 'Upper_CI']
model_comparison_results.csv -> (15, 4) ['CategoryName', 'Model', 'MAPE', 'RMSE']
best_model_per_category.csv -> (5, 4) ['CategoryName', 'Model', 'MAPE', 'RMSE']


## Step 2 — Known data-quality issues (checked here so the app can handle them gracefully)

1. **MAPE is unusable for model selection.** Some test-period months have zero actual demand, which makes MAPE (a percentage error) divide by zero and blow up to huge numbers (~1e19). `best_model_per_category.csv` correctly selects by RMSE instead — the app follows the same rule and does not display MAPE as a selection metric.
2. **`future_demand_forecast.csv` stores only one fixed confidence level (95%).** To power the confidence-interval slider without re-fitting models inside the app, we back out the implied standard error from the stored 95% band, then rescale it for whichever confidence % the user picks, using the normal-distribution z-score. This is a standard approximation, not a re-estimation — documented clearly in the app's UI.
3. **Mother Board and RAM have extremely wide/negative confidence intervals**, indicating the underlying forecast model did not converge well for those categories. The app clips negative bounds at 0 for display and shows an explicit warning banner rather than hiding the issue.

In [3]:
print("MAPE range across all rows:")
print(model_comparison_df['MAPE'].describe())
print()
print("Best model per category (selected by RMSE):")
print(best_model_df)

MAPE range across all rows:
count    1.500000e+01
mean     3.946370e+19
std      2.908734e+19
min      6.666667e+01
25%      2.683759e+19
50%      3.588717e+19
75%      4.270186e+19
max      1.236193e+20
Name: MAPE, dtype: float64

Best model per category (selected by RMSE):
   CategoryName           Model          MAPE        RMSE
0           CPU          SARIMA  1.952404e+19   93.922294
1  Mother Board  Moving Average  6.666667e+01  127.432335
2           RAM  Moving Average  9.369629e+18   34.863232
3       Storage  Moving Average  2.727396e+19   97.778041
4    Video Card  Moving Average  3.487788e+19  121.350128


In [4]:
from scipy.stats import norm

def get_adjusted_ci(cat_future, confidence_pct):
    """Rescale the stored 95% CI to any requested confidence level.
    Same logic used inside app.py's Future Forecast tab."""
    z_95 = norm.ppf(0.975)
    implied_se = (cat_future['Upper_CI'] - cat_future['Forecast']) / z_95
    z_selected = norm.ppf(0.5 + confidence_pct / 200)
    lower = cat_future['Forecast'] - z_selected * implied_se
    upper = cat_future['Forecast'] + z_selected * implied_se
    return lower.clip(lower=0), upper

# Sanity check across all categories and a range of confidence levels
for cat in historical_df.columns:
    cat_future = future_df[future_df['CategoryName'] == cat].sort_values('OrderDate')
    for conf in [50, 80, 95, 99]:
        lower, upper = get_adjusted_ci(cat_future, conf)
    is_unstable = (cat_future['Upper_CI'] - cat_future['Lower_CI']).max() > 10 * historical_df[cat].max()
    print(f"{cat:15s} | unstable_flag={is_unstable}")

CPU             | unstable_flag=False
Mother Board    | unstable_flag=True
RAM             | unstable_flag=True
Storage         | unstable_flag=False
Video Card      | unstable_flag=False


## Step 3 — Write the Streamlit app to disk

Run this cell to (re)generate `app.py`. `%%writefile` writes everything below it straight to a file — it will overwrite `app.py` if it already exists.

In [5]:
# NOTE: app.py has already been written for you and sits alongside this notebook.
# If you need to regenerate it from scratch, copy its contents here behind a
# %%writefile app.py magic command and re-run this cell.
import os
print('app.py present:', os.path.exists('app.py'))
print('requirements.txt present:', os.path.exists('requirements.txt'))

app.py present: False
requirements.txt present: True


## Step 4 — Run the app

Streamlit does **not** run inside a notebook cell — it needs its own terminal process and opens in a browser tab.

1. Open a **Terminal** (JupyterLab: File → New → Terminal)
2. `cd` into this project folder (same folder as `app.py`)
3. Run:
   ```bash
   pip install -r requirements.txt
   streamlit run app.py
   ```
4. Open the printed `http://localhost:8501` link in your browser
5. Use the sidebar dropdown to switch product categories, and the slider on the **Future Forecast** tab to adjust the confidence interval
6. Stop the app with `Ctrl+C` in the terminal when done

## Step 5 — Deploy to Streamlit Community Cloud (free, public URL)

1. Push this project folder to a GitHub repo (must include `app.py`, `requirements.txt`, and all five CSV files above)
2. Go to https://streamlit.io/cloud and sign in with GitHub
3. Click **New app**, select your repo/branch, and set the main file path to `app.py`
4. Click **Deploy** — Streamlit Cloud installs `requirements.txt` automatically and gives you a public URL
5. Add that URL to your GitHub README so the team lead and reviewers can open it directly